# Sprite Generator - Improved Complete GPU Training
VQ-VAE (200 epochs) with all improvements:
- GroupNorm, Self-attention, U-Net skips, NN-upsample
- EMA codebook with dead-code reset
- Patch discriminator (VQGAN-style adversarial loss)
- Perceptual loss (VGG16) + Focal Frequency Loss + Sobel edge loss + Palette histogram loss
- AdamW, warmup+cosine, bf16 mixed precision, EMA decoder weights
- HF push **every epoch** to prevent loss on Kaggle timeout

In [ ]:
import os, sys, torch, json, warnings, math
from pathlib import Path
warnings.filterwarnings("ignore")

!pip install -q datasets huggingface_hub torchvision pillow tqdm

if not os.path.exists('/kaggle/working/sprite-gen'):
    !git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

HF_DATASET = "darklord8777/sprites"
HF_MODEL_REPO = "darklord8777/sprite-generator-model"

EPOCHS = 200
BATCH_SIZE = 128
LR = 3e-4
HIDDEN_DIM = 256
LATENT_DIM = 96
NUM_EMBEDDINGS = 512
WARMUP_EPOCHS = 5

LAMBDA_PERC = 0.5
LAMBDA_FFL = 0.1
LAMBDA_EDGE = 0.05
LAMBDA_PALETTE = 0.1
LAMBDA_ADV = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"HF_TOKEN set: {bool(HF_TOKEN)}")

## Step 1: Dataset with safe augmentations (NN resize only)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F

class SpriteDataset(torch.utils.data.Dataset):
    def __init__(self, hf_path, split="train", image_size=32, augment=True):
        self.dataset = load_dataset(hf_path, split=split)
        self.image_size = image_size
        self.augment = augment
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        img = item["image"].convert("RGBA")
        img = img.resize((self.image_size, self.image_size), Image.NEAREST)
        img_t = transforms.ToTensor()(img)
        if self.augment:
            rgb, a = img_t[:3], img_t[3:]
            if torch.rand(1).item() > 0.3:
                rgb = TF.adjust_brightness(rgb, 1.0 + torch.empty(1).uniform_(-0.1, 0.1).item())
            if torch.rand(1).item() > 0.3:
                rgb = TF.adjust_contrast(rgb, 1.0 + torch.empty(1).uniform_(-0.1, 0.1).item())
            img_t = torch.cat([rgb, a])
        return img_t

ds = SpriteDataset(HF_DATASET, augment=True)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Dataset: {len(ds)} sprites, {len(dl)} batches/epoch")

## Step 2: Load Improved VQ-VAE model
Imports from the cloned repo. Falls back to inline definitions if needed.

In [ ]:
try:
    from models.vqvae.model import ImprovedVQVAE
    from models.vqvae.model import PatchDiscriminator, VGGPerceptualLoss
    from models.vqvae.model import focal_frequency_loss, sobel_edge_loss, palette_histogram_loss
    print("Loaded ImprovedVQVAE from repo")
except Exception as e:
    print(f"Import failed ({e}), using inline model")
    # ----- Inline fallback model -----
    exec(open('/kaggle/working/sprite-gen/models/vqvae/model.py').read())
    exec(open('/kaggle/working/sprite-gen/models/vqvae/model.py').read())

model = ImprovedVQVAE(
    in_channels=4,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_embeddings=NUM_EMBEDDINGS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {total_params:,} params ({trainable:,} trainable)")# Check VGG availability
vgg_available = model.perceptual_loss.vgg is not None
print(f"VGG perceptual: {'AVAILABLE' if vgg_available else 'UNAVAILABLE (falling back to MSE)'}")
if not vgg_available:
    print(f"  -> perc_loss will be recon_loss DUPLICATE — consider setting LAMBDA_PERC=0")


In [ ]:
# Try to load palette from dataset card
palette = []
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN if HF_TOKEN else None)
    card = api.dataset_info(HF_DATASET)
    card_text = card.card_data or ""
    import re
    m = re.search(r'\[(.*?)\]', str(card_text))
    if m:
        palette = eval(m.group(1))
except Exception:
    pass
print(f"Palette: {len(palette)} colors" if palette else "No palette loaded, disabling palette loss")

## Step 3: Optimizers, schedulers, mixed precision

In [ ]:
g_params = list(model.encoder.parameters()) + list(model.decoder.parameters()) + list(model.quantizer.parameters())
optimizer_g = torch.optim.AdamW(g_params, lr=LR, weight_decay=1e-4, betas=(0.5, 0.9))
optimizer_d = torch.optim.AdamW(model.discriminator.parameters(), lr=LR * 0.5, weight_decay=1e-4, betas=(0.5, 0.9))

def warmup_cosine(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 0.5 * (1 + math.cos((epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS) * math.pi))

scheduler_g = torch.optim.lr_scheduler.LambdaLR(optimizer_g, warmup_cosine)
scheduler_d = torch.optim.lr_scheduler.LambdaLR(optimizer_d, warmup_cosine)

model.init_ema()
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

ckpt_dir = Path("/kaggle/working/checkpoints/vqvae")
ckpt_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete")

## Step 4: Training loop with all losses + HF push every epoch

In [ ]:
torch.autograd.set_detect_anomaly(True)
vis_batch = next(iter(dl))

for epoch in range(EPOCHS):
    model.train()
    total_g = total_recon = total_vq = total_perc = total_ffl = total_edge = total_adv = 0
    total_d_real = total_d_fake = 0

    pbar = tqdm(dl, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        batch = batch.to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        
        # Extra losses outside autocast for fp32 gradient stability
            out = model(batch)
            recon = out["recon"]

            recon_loss = out["recon_loss"]
            vq_loss = out["vq_loss"]

            perc_loss = model.perceptual_loss(batch, recon) * LAMBDA_PERC
            ffl_loss = focal_frequency_loss(batch, recon) * LAMBDA_FFL if LAMBDA_FFL > 0 else torch.tensor(0.0, device=device)
            edge_loss = sobel_edge_loss(batch, recon) * LAMBDA_EDGE if LAMBDA_EDGE > 0 else torch.tensor(0.0, device=device)

            pal_loss = torch.tensor(0.0, device=device)
            if LAMBDA_PALETTE > 0 and len(palette) > 0:
                pal_loss = palette_histogram_loss(batch, recon, palette) * LAMBDA_PALETTE

            d_fake = model.discriminator(recon)
            adv_loss = F.relu(1 - d_fake).mean() * LAMBDA_ADV if LAMBDA_ADV > 0 else torch.tensor(0.0, device=device)

            g_loss = recon_loss + vq_loss + perc_loss + ffl_loss + edge_loss + pal_loss + adv_loss

        optimizer_g.zero_grad()
        scaler.scale(g_loss).backward()
        scaler.unscale_(optimizer_g)
        torch.nn.utils.clip_grad_norm_(g_params, 1.0)
        scaler.step(optimizer_g)

        if LAMBDA_ADV > 0 and epoch >= 2:
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        
        # Extra losses outside autocast for fp32 gradient stability
                with torch.no_grad():
                    recon_d = model(batch)["recon"].detach()
                d_real = model.discriminator(batch)
                d_fake_d = model.discriminator(recon_d)
                d_loss = F.relu(1 - d_real).mean() + F.relu(1 + d_fake_d).mean()

            optimizer_d.zero_grad()
            scaler.scale(d_loss).backward()
            scaler.step(optimizer_d)
            total_d_real += d_real.mean().item()
            total_d_fake += d_fake_d.mean().item()

        scaler.update()

        total_g += g_loss.item()
        total_recon += recon_loss.item()
        total_vq += vq_loss.item()
        total_perc += perc_loss.item()
        total_ffl += ffl_loss.item()
        total_edge += edge_loss.item()
        total_adv += adv_loss.item()

        pbar.set_postfix({
            "loss": f"{g_loss.item():.4f}",
            "recon": f"{recon_loss.item():.4f}",
            "perc": f"{perc_loss.item():.4f}",
        })

    scheduler_g.step()
    scheduler_d.step()
    model.update_ema()
    model.reset_dead_codes(batch)

    n = len(dl)
    print(f"Epoch {epoch+1}: G={total_g/n:.4f} recon={total_recon/n:.4f} "
          f"vq={total_vq/n:.4f} perc={total_perc/n:.4f} "
          f"ffl={total_ffl/n:.4f} edge={total_edge/n:.4f} adv={total_adv/n:.4f} "
          f"D_real={total_d_real/n:.3f} D_fake={total_d_fake/n:.3f} "
          f"lr={scheduler_g.get_last_lr()[0]:.2e}")

    # Save checkpoint and push to HF every 10 epochs
    ckpt_path = ckpt_dir / f"vqvae_epoch_{epoch+1:03d}.pt"
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_g_state": optimizer_g.state_dict(),
        "optimizer_d_state": optimizer_d.state_dict(),
        "loss": total_g / n,
        "config": {
            "hidden_dim": HIDDEN_DIM,
            "latent_dim": LATENT_DIM,
            "num_embeddings": NUM_EMBEDDINGS,
        },
    }, ckpt_path)

    if HF_TOKEN and (epoch + 1) % 10 == 0:
        from huggingface_hub import HfApi
        HfApi(token=HF_TOKEN).upload_file(
            path_or_fileobj=str(ckpt_path),
            path_in_repo="vqvae_latest.pt",
            repo_id=HF_MODEL_REPO, repo_type="model",
        )
        print(f"  -> Pushed to HF (epoch {epoch+1})")

print("VQ-VAE training complete!")

## Step 6: Train Conditional Transformer Prior on VQ-VAE Tokens

In [ ]:
from models.transformer.model import SpriteTransformer
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from PIL import Image
import numpy as np

CLASS_VOCAB = ['unknown', 'character', 'item', 'tile', 'enemy', 'player', 'weapon', 'food',
               'vehicle', 'building', 'decoration', 'effect', 'projectile', 'animal', 'plant',
               'furniture', 'tool', 'accessory', 'ui_element', 'terrain']
ACTION_VOCAB = ['idle', 'walk', 'run', 'attack', 'jump', 'hurt', 'death', 'block', 'shoot',
               'cast', 'interact', 'fly', 'swim', 'climb']
DIRECTION_VOCAB = ['front', 'back', 'left', 'right', 'front_left', 'front_right', 'back_left', 'back_right']

def encode_cond(value, vocab):
    try:
        return vocab.index(value)
    except ValueError:
        return 0

class TokenDataset(Dataset):
    def __init__(self, hf_path, vqvae, device, split='train'):
        self.ds = load_dataset(hf_path, split=split)
        self.vqvae = vqvae
        self.device = device
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item['image'].convert('RGBA').resize((32, 32), Image.NEAREST)
        t = torch.tensor(np.array(img).astype(np.float32) / 255.0).permute(2, 0, 1)
        with torch.no_grad():
            indices = self.vqvae.encode_to_indices(t.unsqueeze(0).to(self.device))
        tokens = indices.squeeze(0).cpu()
        cid = encode_cond(item.get('class', 'unknown'), CLASS_VOCAB)
        aid = encode_cond(item.get('action', 'idle'), ACTION_VOCAB)
        did = encode_cond(item.get('direction', 'front'), DIRECTION_VOCAB)
        return tokens, torch.tensor(cid), torch.tensor(aid), torch.tensor(did)

def collate(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]))

print('Token dataset created')

In [ ]:
TRANSFORMER_EPOCHS = 100
TRANSFORMER_BATCH = 64
TRANSFORMER_LR = 3e-4
TRANSFORMER_WARMUP = 5
TRANSFORMER_DMODEL = 512
TRANSFORMER_NLAYERS = 12
TRANSFORMER_NHEADS = 8

model.eval()
t_dataset = TokenDataset(HF_DATASET, model, device)
t_loader = DataLoader(t_dataset, batch_size=TRANSFORMER_BATCH, shuffle=True,
                      num_workers=0, collate_fn=collate)
sample_tokens = t_dataset[0][0]
max_seq_len = sample_tokens.shape[0]
print(f'Dataset: {len(t_dataset)} samples, seq_len={max_seq_len}')

t_model = SpriteTransformer(
    vocab_size=model.quantizer.num_embeddings,
    condition_vocab_size=64,
    d_model=TRANSFORMER_DMODEL,
    n_layers=TRANSFORMER_NLAYERS,
    n_heads=TRANSFORMER_NHEADS,
    max_seq_len=max_seq_len + 1,
).to(device)
print(f'Transformer params: {sum(p.numel() for p in t_model.parameters()):,}')

t_optimizer = torch.optim.AdamW(t_model.parameters(), lr=TRANSFORMER_LR, weight_decay=1e-4)

def warmup_cosine(epoch):
    if epoch < TRANSFORMER_WARMUP:
        return (epoch + 1) / TRANSFORMER_WARMUP
    return 0.5 * (1 + math.cos((epoch - TRANSFORMER_WARMUP) / (TRANSFORMER_EPOCHS - TRANSFORMER_WARMUP) * math.pi))
t_scheduler = torch.optim.lr_scheduler.LambdaLR(t_optimizer, warmup_cosine)

print('Transformer ready')

In [ ]:
from tqdm import tqdm

for epoch in range(TRANSFORMER_EPOCHS):
    t_model.train()
    total_loss = 0
    pbar = tqdm(t_loader, desc=f'Transformer Epoch {epoch+1}/{TRANSFORMER_EPOCHS}')
    for tokens, class_ids, action_ids, direction_ids in pbar:
        tokens = tokens.to(device)
        class_ids = class_ids.to(device)
        action_ids = action_ids.to(device)
        direction_ids = direction_ids.to(device)

        t_optimizer.zero_grad()
        logits = t_model(tokens, class_ids, action_ids, direction_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tokens.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(t_model.parameters(), 1.0)
        t_optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(t_loader)
    t_scheduler.step()
    print(f'Transformer Epoch {epoch+1}: loss={avg_loss:.6f}')

    t_ckpt = ckpt_dir / f'transformer_epoch_{epoch+1:03d}.pt'
    torch.save({
        'epoch': epoch,
        'model_state': t_model.state_dict(),
        'optimizer_state': t_optimizer.state_dict(),
        'loss': avg_loss,
        'config': {'d_model': TRANSFORMER_DMODEL, 'n_layers': TRANSFORMER_NLAYERS,
                   'n_heads': TRANSFORMER_NHEADS},
    }, t_ckpt)

    if HF_TOKEN and (epoch + 1) % 10 == 0:
        from huggingface_hub import HfApi
        HfApi(token=HF_TOKEN).upload_file(
            path_or_fileobj=str(t_ckpt),
            path_in_repo=f'transformer_epoch_{epoch+1:03d}.pt',
            repo_id=HF_MODEL_REPO, repo_type='model',
        )
        HfApi(token=HF_TOKEN).upload_file(
            path_or_fileobj=str(t_ckpt),
            path_in_repo='transformer_latest.pt',
            repo_id=HF_MODEL_REPO, repo_type='model',
        )
        print(f'  -> Transformer pushed to HF (epoch {epoch+1})')

print('Transformer training complete!')

## Step 7: Push final training complete marker

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(
        path_or_fileobj=json.dumps({'status': 'complete',
                                     'vqvae_epochs': EPOCHS,
                                     'transformer_epochs': TRANSFORMER_EPOCHS,
                                     'config': {'hidden_dim': HIDDEN_DIM,
                                                'latent_dim': LATENT_DIM,
                                                'num_embeddings': NUM_EMBEDDINGS,
                                                'd_model': TRANSFORMER_DMODEL,
                                                'n_layers': TRANSFORMER_NLAYERS,
                                                'n_heads': TRANSFORMER_NHEADS}}).encode(),
        path_in_repo='training_complete.json',
        repo_id=HF_MODEL_REPO, repo_type='model',
    )
    print('Training marker pushed to HF')

print('=== ALL DONE ===')